In [1]:
import numpy as np
import torch
from tqdm.notebook import tqdm

In [13]:
data_train = np.loadtxt("data/FordA_TRAIN.txt")
X_train = data_train[:, 1:]
y_train = data_train[:, 0].astype(int)

data_test = np.loadtxt("data/FordA_TEST.txt")
X_test = data_test[:, 1:]
y_test = data_test[:, 0].astype(int)

print(X_train.shape)  # (3601, 500)
print(y_train.shape)  # (3601,)

(3601, 500)
(3601,)


In [14]:
X_train[:, np.newaxis, :].shape

(3601, 1, 500)

In [21]:
train_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_train[:, np.newaxis, :], dtype=torch.float32), torch.tensor((y_train + 1) / 2, dtype=torch.float32)
)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)

test_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_test[:, np.newaxis, :], dtype=torch.float32), torch.tensor((y_test + 1) / 2, dtype=torch.float32)
)
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False)

In [22]:
EPOCHS = 25
device="cuda" if torch.cuda.is_available() else "cpu"

In [23]:
def train_step(model, train_dl, loss_fn, optimizer, device):
    model.train()
    train_loss = 0.0

    for inputs, labels in tqdm(train_dl, desc="Training", total=len(train_dl), leave=False):
        optimizer.zero_grad()

        inputs = inputs.to(device)
        labels = labels.to(device).long()   # важно для CrossEntropyLoss

        outputs = model(inputs)             # shape: [batch_size, 2]
        loss = loss_fn(outputs, labels)     # labels: [batch_size]

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(train_dl)


def eval_step(model, test_dl, loss_fn, device):
    model.eval()
    test_loss = 0.0

    with torch.no_grad():
        for inputs, labels in tqdm(test_dl, desc="Evaluating", total=len(test_dl), leave=False):
            inputs = inputs.to(device)
            labels = labels.to(device).long()

            outputs = model(inputs)         # shape: [batch_size, 2]
            loss = loss_fn(outputs, labels)

            test_loss += loss.item()

    return test_loss / len(test_dl)


def train(model, train_dl, test_dl, device, EPOCHS):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = torch.nn.CrossEntropyLoss()

    for epoch in tqdm(range(EPOCHS), desc="Epochs", total=EPOCHS, leave=False):
        train_loss = train_step(model, train_dl, loss_fn, optimizer, device)
        test_loss = eval_step(model, test_dl, loss_fn, device)

        print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

    return model

In [24]:
from model import CNNModel

model = CNNModel(input_size=1).to(device)
model

CNNModel(
  (net): Sequential(
    (0): Conv1d(1, 32, kernel_size=(8,), stride=(1,), padding=(4,))
    (1): ReLU()
    (2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (3): ReLU()
    (4): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): Dropout(p=0.25, inplace=False)
    (6): ReLU()
    (7): AdaptiveAvgPool1d(output_size=1)
  )
  (ffn): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [26]:
train(model, train_dl=train_dl, test_dl=test_dl, device=device, EPOCHS=25)

Epochs:   0%|          | 0/25 [00:00<?, ?it/s]

Training:   0%|          | 0/113 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 1/25, Train Loss: 0.6220, Test Loss: 0.5189


Training:   0%|          | 0/113 [00:00<?, ?it/s]

...

Epoch 24/25, Train Loss: 0.2337, Test Loss: 0.2297


Training:   0%|          | 0/113 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 25/25, Train Loss: 0.2316, Test Loss: 0.2376


CNNModel(
  (net): Sequential(
    (0): Conv1d(1, 32, kernel_size=(8,), stride=(1,), padding=(4,))
    (1): ReLU()
    (2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (3): ReLU()
    (4): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): Dropout(p=0.25, inplace=False)
    (6): ReLU()
    (7): AdaptiveAvgPool1d(output_size=1)
  )
  (ffn): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [27]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report
import torch
import numpy as np

def evaluate(model, dataloader, device):
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            logits = model(inputs).cpu()   # shape: [batch_size, 2]
            all_logits.append(logits)
            all_labels.append(labels)

    all_logits = torch.cat(all_logits, dim=0)          # [N, 2]
    all_labels = torch.cat(all_labels, dim=0).numpy().astype(int)  # [N]

    probs = torch.softmax(all_logits, dim=1).numpy()   # [N, 2]
    preds = np.argmax(probs, axis=1)                   # [N]
    pos_probs = probs[:, 1]                            # вероятность класса 1

    print(f"Accuracy:   {accuracy_score(all_labels, preds):.4f}")
    print(f"ROC-AUC:    {roc_auc_score(all_labels, pos_probs):.4f}")
    print(f"F1 (macro): {f1_score(all_labels, preds, average='macro'):.4f}")
    print()
    print(classification_report(all_labels, preds, target_names=["Class 0", "Class 1"]))

print("=== Test ===")
evaluate(model, test_dl, device)

=== Test ===
Accuracy:   0.9076
ROC-AUC:    0.9706
F1 (macro): 0.9076

              precision    recall  f1-score   support

     Class 0       0.95      0.87      0.91       681
     Class 1       0.87      0.95      0.91       639

    accuracy                           0.91      1320
   macro avg       0.91      0.91      0.91      1320
weighted avg       0.91      0.91      0.91      1320



In [28]:
torch.save(model.state_dict(), 'time_series_cnn.pt')